<a href="https://colab.research.google.com/github/YzhangBrian/BookCode/blob/main/Yujie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COM3502-4502-6502 Speech Processing - Python Programming Assignment

## General Information

This programming assignment is worth $55$% of the overall course mark.

You are free to complete this assignment in your own time. However, feedback, advice and guidance are available during the lab classes and via the discussion board on Blackboard.

Note: Via these channels, we try to help you as much as possible, but will not debug your code or provide solutions to the assignment itself.

Note: It will take some time to complete this assignment, so plan your work accordingly over the coming weeks. Read these instructions carefully.

Note: Please be aware that students registered on COM4502 and COM6502 have **additional tasks** to perform. These are marked ‘COM4502-6502 Only’.

Note: You should always ensure that your results (e.g. in terms of plots you create) are clear to understand and leave no room for misinterpretation. This can often be easily achieved by adding proper $x$- and $y$-axis labels, titles, legends etc. Where results are not clear to interpret, this might result in missed points.


## Student Data

Student Family Name: <span style="font-weight:bold;color:orange">**Yujie**</span>

Student Given Name(s): <span style="font-weight:bold;color:orange">**Zhang**</span>

Date of submission: <span style="font-weight:bold;color:orange">**16-12-2025**</span>

## Copyright

This programming assignment is part of the lecture COM[3502](http://www.dcs.shef.ac.uk/intranet/teaching/public/modules/level3/com3502.html "Open web page for COM3502 module")-[4502](http://www.dcs.shef.ac.uk/intranet/teaching/public/modules/level4/com4502.html "Open web page for COM4502 module")-[6502](http://www.dcs.shef.ac.uk/intranet/teaching/public/modules/msc/com6502.html "Open web page for COM4502 module") Speech Processing at the [University of Sheffield](https://www.sheffield.ac.uk/ "Open web page of The University of Sheffield"), [School of Computer Science](https://www.sheffield.ac.uk/cs "Open web page of School of Computer Science"), University of Sheffield.


This notebook is licensed as an assignment to be used during the lecture COM3502-4502-6502 Speech Processing at the University of Sheffield. Any further use is only permitted if agreed with the [module lead](mailto:s.goetze@sheffield.ac.uk).

It should be a matter of course that rules of [unfair means](https://www.sheffield.ac.uk/apse/apo/quality/assessment/unfair) apply and the assignment is not to be shared with or made available to other persons besides those participating in the module during the same academic year. This includes publishing on web pages etc. All questions can be asked during the lab classes or using the Blackboard Discussion board.


## Hand-In Procedure and Deadline

Once you have completed the assignment you should submit a `.zip` file (via Blackboard) containing your solution (as a file named `YourFamilyName.ipynb`) and possibly other sources linked in your Jupyter Notebook. Also, the `.zip` filename should be of the form `YourFamilyName.zip`. Please also ensure that your name is entered correctly in the section above.

Standard school penalties apply for late hand-in and plagiarism.

The **deadline** for handing-in this assignment (via Blackboard) is
<span style="font-weight:bold;color:red">**15:00 on Tuesday, 16th December 2025**</span>.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 0:**
    
Ensure that you data is correctly entered in the section at the top of this sheet and that the filename is in the form `YourFamilyName.ipynb`.
    
</div>

## Libraries

You should be familiar with the use of the following Python libraries from the lab. You should not need to use additional ones. You are allowed to use additional libraries if necessary for your code. If they need to be installed by `!pip install <libraryname>` or `!conda install <libraryname>`, please indicate this as a comment in your code. You should not make use of libraries that can't be installed by either `!pip install` or `!conda install`. You must ensure that your Notebook runs "out of the box". You can test this on the Computer Lab machines in the Diamond if you are unsure and using your own computer.

In [10]:
#Let's do some necessary and nice-to-have imports
%matplotlib inline
import matplotlib.pyplot as plt    # plotting
#import seaborn as sns; sns.set()  # styling
import numpy as np                 # math

import soundfile as sf             # to load files
from IPython import display as ipd # for sound playback

from scipy import signal           # filter designs (if not already imported)

import os
from pathlib import Path
import urllib.request

# Download, load, and analyse audio

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T1:**
    
* Load a wave file containing speech. You can find a file at <a href="https://staffwww.dcs.shef.ac.uk/people/n.ma/comx502/speech.wav">https://staffwww.dcs.shef.ac.uk/people/n.ma/comx502/speech.wav</a> and should be able to download this. You can also use your own WAVE files if you prefer this. If you want to record WAVE files and are using your own computer, the program [Audacity](https://www.audacityteam.org/) is one possibility to [record WAVE files](https://manual.audacityteam.org/man/basic_recording_editing_and_exporting.html).
    
* Visualise the signal in time domain, in the spectral domain (as spectrum) and as a time-frequency representation (spectrogram). Please ensure proper axis labels for all your plot in this assignment.

* Playback the signal.
    
</div>

In [11]:
# your code here
#
COURSE_URL = "https://staffwww.dcs.shef.ac.uk/people/n.ma/comx502/speech.wav"
LOCAL_CANDIDATES = [Path("data/music_44k.wav"), Path("data/speech.wav")]

# 选择本地已有的音频；都没有就下载课程文件到 data/speech.wav
chosen = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
if chosen is None:
    try:
        print("Local WAV not found; downloading course speech.wav ...")
        urllib.request.urlretrieve(COURSE_URL, "data/speech.wav")
        chosen = Path("data/speech.wav")
        print("Saved to data/speech.wav")
    except Exception as e:
        raise RuntimeError("请将一段 WAV 放到 data/music_44k.wav 或 data/speech.wav 再运行。") from e

print("Using audio:", chosen)

# 读取
x, fs = sf.read(str(chosen), always_2d=False)
print("fs =", fs, "Hz | shape:", x.shape, "| dtype:", x.dtype)

# 立体声 -> 单声道
if x.ndim == 2:
    x = x.mean(axis=1)
    print("Converted to mono:", x.shape)

# 若为整型，转 float32
if x.dtype.kind in "iu":
    x = x.astype(np.float32) / np.iinfo(x.dtype).max
else:
    x = x.astype(np.float32)

N = len(x)
duration = N / fs
t = np.arange(N) / fs
print(f"Duration: {duration:.2f}s  ({N} samples)")


Local WAV not found; downloading course speech.wav ...


RuntimeError: 请将一段 WAV 放到 data/music_44k.wav 或 data/speech.wav 再运行。

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q1:**

* Determine the sampling frequency $f_s$ in Hz and the length of the signal in seconds. What is the sampling interval $T_s$ of your signal? What is the highest occurring frequency?
    
Note: You can either give your answer in the form of a code block (e.g. by using the `print()` functions) or as text. For the latter, change the [type of the next cell](https://jupyter-notebook.readthedocs.io/en/stable/notebook.html#structure-of-a-notebook-document) from `code` to `markdown` or use the yellow example text below.
    
</div>

In [ ]:
# Your answer here
#
# ...

<span style="font-weight:bold;color:orange">In case you want to answer by written text, we would appreciate if you  colour-code your answers, e.g. like using orange font colour as illustrated in this example. This helps us, not to overlook parts of your answers.</span>

## Signal Analysis

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T2:**
    
* Generate a synthetic audio signal consisting of three sinusoids with frequencies of $100$ Hz, $300$ Hz, and $500$ Hz including an initial phase which should be different from zero at least for one of the sinusoids. Assume a sampling rate of $8$ kHz and a duration of $2$ seconds. Write code to generate and plot the waveform of this signal. Compute and plot the magnitude spectrum of the signal.
    
* Use the Fast Fourier Transform (FFT) to calculate the spectrum, and display only the positive frequencies. Identify the main frequency components in the signal (using Python code) based on the magnitude spectrum and briefly explain your observations and identify the three main frequency peaks (as written answer below).

</div>

In [ ]:
# Your answer here
#
# ...

<span style="font-weight:bold;color:orange">... Your answer here ...</span>

# Piece-wise linear filtering in the time domain

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T3:**
    
* Design a high-pass filter with a cut-off frequency of $\approx 500$ Hz using a filter design method of your choice.
* Design a low-pass filter with a cut-off frequency of $\approx 500$ Hz.
* Design a band-stop filter with a cut-off frequencies of $\approx 300$ Hz and of $\approx 1.1$ kHz.
* Simulate the effect of a land-line telephone by eliminating all energy below $300$ Hz and above $3,400$ Hz.
* Visualise the transfer functions of the filters and the zero-pole plots.
* Apply the designed filters, compare filter input and output as a time-frequency visualisation and play back the filtered signal.
    
Note: Don't forget proper labeling/description of your figures to make clear what is what.
    
Note: In case you encounter stability problems, remember that we mentioned in the lecture, that filters can be designed as second-order-systems (SOS) which the design methods you are familiar with can realise.
</div>

In [ ]:
# Your code here
#
# ...

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q2:**

* Explain the behaviour of the designed band-stop filter, i.e. describe (briefly) what you can see in the generated plots. If you didn't generate plots you can explain what you would expect to see.

    
</div>

**Answer to Question Q2:**

<span style="font-weight:bold;color:orange">
    ... your answer here ...
</span>

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q3:**
    
* Which sounds are most affected when the low-pass cut-off frequency is set to around $500$
Hz - vowels or consonants - and why?
    
</div>

**Answer to Question Q3:**

<span style="font-weight:bold;color:orange">
    ...your answer here ...
</span>

# Audio Effects

## Low-Frequency Oscillator

Many ‘voice effects (FXs)’ are achieved by modifying some characteristic of the speech using a low-frequency oscillator or *LFO*. LFOs typically have two controls: speed (which is specified by the frequency in Hertz) and depth (which specifies the magnitude of the effect). The following tasks will require several LFOs, so it makes sense to implement one in the following.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T4:**
    
* Implement a Low Frequency Oscillator as a function `lfo()` as described below. Visualise that your function works by generating a sine and a square wave of frequency $5$ Hz and length $2$ seconds with different depths.
    
Note: There will be an extra point in the marking if you **do not** use the `scipy` library to solve this task.
    
</div>

In [ ]:
def lfo(speed_hz, depth, num_samples, fs=44100, square_curve=False):
    '''
    Low-frequency oscillator

    Parameters
    ----------
    speed_hz : float
       frequency of generated signal in Hertz
    depth : float
        magnitude of the effect
    num_samples : int
        length of the signal in samples
    fs : float, optional
        sampling frequency in Hz, default 44100
    square_curve : boolean, optional (default: False)
        generate square wave if true, generate sine wave if false

    Example use:
    -------
        sig_square = lfo(speed_hz=5, depth=0.7, num_samples=88200, fs=44100, square_curve=True)
    '''

    # Your code here
    #
    # ...

In [ ]:
# Your code here to show that the LFO works
#
# ...

Although your function outputs audio, you are unlikely to be able to hear it as the frequency is so low. However, you can check that it is functioning correctly by combining it with other audio signals as we will do in the following.

## Amplitude Modulation - Tremolo

*Tremolo* is one of the most basic voice manipulations that makes use of an LFO. In this effect, the amplitude of a speech signal is [modulated](https://en.wikipedia.org/wiki/Amplitude_modulation), i.e. the speech waveform is multiplied by a variable gain that ranges between $1-$ `modulation_depth` and $1$. This means that if the `modulation_depth` equals $1$, the variable gain varies between $1-1=0$ and $1$.

Your LFO outputs an audio signal between `-depth` and `+depth` which is different from the `modulation_depth` above. So, in order to modulate the amplitude of the speech correctly, the output of the LFO has to be scaled and applied appropriately.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T5:**
    
* Implement  a function `tremolo()` using your function `lfo()` and modulate the amplitude of the speech signal.
* Experiment with different settings for `speed` and `modulation_depth`. In particular, note that a square wave with a *speed* between $3$ and $4$ Hz (and `modulation_depth` = $1$) has a very destructive effect on the intelligibility of the output. This is because $3-4$ Hz corresponds to the typical syllabic rate of speech.
* Proof that the effect works by a proper visualisation of the filtered speech signal and describe what can be observed and perceived.
    
</div>

In [ ]:
def tremolo(signal, fs, speed_hz, modulation_depth, square_curve=False):
    '''
    Applies a tremolo effect to a signal

    Parameters
    ----------
    signal : float
       input signal to which the effect should be applied
    fs : int
        sampling frequency in Hz
    speed_hz : float
       frequency of LFO and by this also the effect
    modulation_depth : float
        magnitude of the effect
    square_curve : boolean, optional
        generate square wave if true, generate sine wave if false

    Return
    ----------
    signal after application of tremolo effect

    Example use:
    -------
        signal_tremolo = tremolo(signal=audio_in, fs=fs, speed_hz=10, modulation_depth=1, square_curve=False)
    '''

    # Your code here
    #
    # ...

In [ ]:
# Your code here to show an example of the Tremolo effect
#
# ...

## Ring Modulation

Another basic effect is to multiply the speech signal by the output of an LFO. This is known as ‘ring modulation’.

Note: In the BBC TV series [Dr. Who](https://en.wikipedia.org/wiki/Doctor_Who), the voices of the alien [Daleks](https://en.wikipedia.org/wiki/Dalek) are generated by a ring modulator with an LFO set to around 30 Hz. The voice actors also spoke using a stilted monotonic intonation in order to enhance the effect. You can try this yourself by recording your own voice and applying the effect.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T6 (Ring Modulation):**
    
* Implement a function `ring_modulation()` using your function `lfo()` and modulate the amplitude of the speech signal by multiplying with the LFO signal.
* Experiment with different settings for `speed` and `depth`. Note how the timbre of the resulting sound is subtly different from *tremolo*.
* Proof that the effect works by a proper visualisation of the filtered speech signal and describe what can be observed and perceived.
    
</div>

In [ ]:
def ring_modulation(signal, fs, speed_hz, depth, square_curve=False):
    '''
    Applies a ring modulation effect to a signal

    Parameters
    ----------
    signal : float
       input signal to which the effect should be applied
    fs : int
        sampling frequency in Hz
    speed_hz : float
       frequency of LFO and by this also the effect
    depth : float
        magnitude of the effect
    square_curve : boolean, optional
        generate square wave if true, generate sine wave if false

    Return
    ----------
    signal after application of the ring modulation effect

    Example use:
    -------
        signal_ring_mod = ring_modulation(audio_in, fs, 10, 1, square_curve=False)
    '''
#
# ...

In [ ]:
# Your code here to show an example of the Ring Modulation effect
#
# ...

## Frequency Shifting

Many Vocal FX are the result of altering the frequencies present, e.g. changing the pitch of a voice. There are many algorithms for frequency shifting. You have already implemented an approximate solution with your ring modulator.

For simplicity, the following function will be given implementing frequency shifting.

In [ ]:
# the following code in this cell is taken and slightly adapted from:
# https://gist.github.com/lebedov/4428122

import scipy.signal as sig

def nextpow2(n):
    '''Return the first integer N such that 2**N >= abs(n)'''
    return int(np.ceil(np.log2(np.abs(n))))

def frequency_shift(signal, fs, shift_amount):
    '''
    Shift the specified signal by the specified frequency.

    Parameters
    ----------
    signal : float
       input signal to which the effect should be applied
    fs : int
        sampling frequency in Hz
    shift_amount : float
       amount of frequency shift (in Hz)

    Return
    ----------
    signal after application of the frequency shifting effect

    Example use:
    -------
        signal_frequency_shifted = frequency_shift(audio_in, fs, 100)
    '''

    # Pad the signal with zeros to prevent the FFT invoked by the transform from
    # slowing down the computation:
    N_orig = len(signal)
    N_padded = 2 ** nextpow2(N_orig)
    t = np.arange(0, N_padded)
    return (
        sig.hilbert(
            np.hstack((signal, np.zeros(N_padded - N_orig, signal.dtype)))
        )
        * np.exp(2j * np.pi * shift_amount * t / fs)
    )[:N_orig].real

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T7 (Frequency Shifting):**
    
* Visualise the effect of the frequency shift effect using an appropriate spectral representation.
    
</div>

In [ ]:
# Your code here to show an example of the frequency shift effect
#
# ...

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q4:**

* COM3502-4502-6502: Why can the voice be shifted up in frequency much further than
it can be shifted down in frequency before it becomes severely distorted? Hint: Calculate a spectrum plot if the answer is not immediately clear to you.
* COM4502-6502 ONLY: Your frequency shifter changes all the frequencies present in an input signal. How might it be possible to change the pitch of a voice without altering the formant frequencies?
    
</div>

**Answer to Question Q4:**

<span style="font-weight:bold;color:orange">
    ...your answer here ...
</span>

## Harmony Effect

A classic ‘robotic’ voice can be achieved by simply adding frequency-shifted speech back to the unprocessed original. This effect is known as ‘harmony’. However, rather than simply adding the signals in equal amounts, we will implement a more general-purpose approach.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T8:**
    
* Implement a function `mixer()` that adds the original speech with the manipulated speech in different proportions.
* Implement a function `harmony()` that mixes the input signal with a frequency-shifted version of itself (using the functions `mixer()` and `frequency_shift()`). With your mixer at the $50$-$50$ setting, experiment with different frequency shifts in order to produce the best robotic-sounding output. Report "your optimal" setting.
</div>

In [ ]:
# def mixer(signal1, signal2, percentage_l=0.5):

# Your code here to implement mixing of two signals, used later for the harmony effect
#
# ...

In [ ]:
# def harmony(...

# Your code here to implement the harmony effect
#
# ...

In [ ]:
# Your code here to show an example of the harmony effect
#
# ...

**Answer to question in Task T7:**

<span style="font-weight:bold;color:orange">
    ...your answer here ... <br>
  My optimal setting is ..... <br>
    because .....
</span>

## Frequency Modulation: Vibrato

Now that you have the ability to shift the frequencies in a speech signal, it is very easy to implement another common voice manipulation technique - *vibrato*. All that is required is for the frequency shifter to be controlled by the output of an LFO.



<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T9:**
    
* Implement a function `vibrato()` by connecting an LFO to your frequency shifter, and experiment with different values for speed and depth. Note that the LFO output will need to be scaled to provide an appropriate frequency shift range and then added to the output of the frequency shift.
    
</div>



In [ ]:
# def vibrato(...

# Your code here to show an example of the harmony effect
#
# ...

In [ ]:
# Your code here to show an example of the vibrato effect
#
# ...

## Time Delay Effect - Echo and Comb Filter

Many interesting voice FX can be achieved by delaying the signal and recombining it with itself.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T10:**
    
* Implement a function `echo()` which mixes a signal $s(t)$ with itself in a delayed version, i.e. $s(t-t_0)$. Experiment with various values for the delay $t_0$, and note the different effects you can achieve with delays
  * below $20$ msecs
  * between $20$ and $100$ msecs, and
  * above $100$ msecs.
</div>

In [ ]:
# def echo(...

# Your code here to show an example of the echo effect
#
# ...

In [ ]:
# Your code here to show an example of the echo effect
#
# ...

## Comb Filtering

You should observe that with delays below $20$ msec in your function `echo()`, the signals combine to create a subtle ‘phasing’ effect. This is known as ‘comb filtering’ as the signal is effectively interfering with itself, and frequency components corresponding to multiples of the delay time are enhanced or cancelled out (due to ‘superposition’). Delays between $20$ and $100$ msecs give the effect of the voice being in a reverberant room. Delays above $100$ msecs sound like distant echoes.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T11:**
    
* Visualise the impulse response of your function `echo()` as well as the transfer function. Can you give an explanation from having a look at the transfer function, why this effect would be called *comb filter*?
</div>

In [ ]:
# Your code here
#
# ...

**Answer to question in Task T10:**

<span style="font-weight:bold;color:orange">
    ...your answer here ...
</span>

## Flanger

It is possible to use an LFO to vary the delay. The resulting effect is known as a *flanger*.

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T12:**
    
* Add an LFO to your ‘delay’ to create a ‘flanger’, and experiment with different settings. Note that you will need to scale the output of the LFO, and you will get different effects depending on whether the delayed signal is mixed with the original or not.
</div>

In [ ]:
# Your code here
#
# ...

In [ ]:
# Your code here
#
# ...

# Frequency Analysis

<br>
<div style="border: 2px solid #999; padding: 10px; background: #5c5c9eff;">
    
**Question Q5:**

* COM3502-4502-6502:
    * What does FFT stand for and what does an FFT do?
* COM4502-6502 ONLY: What is a DFT and how is it different from an FFT?
    
</div>

**Answer to Question Q5:**

<span style="font-weight:bold;color:orange">
    ...your answer here ...
</span>

## Creating the Spectrogram Step-by-Step (Task T13 for COM4502-6502 only, Task T14 for all students)

The magnitude $|X[n, \ell]|$ of the STFT for all $n$ and $\ell$ is known as the [spectrogram](https://en.wikipedia.org/wiki/Spectrogram) of a signal. It is frequently used to analyse signals in the time-frequency domain, for instance by a [spectrum analyser](https://en.wikipedia.org/wiki/Spectrum_analyzer). It can be interpreted as a *image* of the signal with (block) time direction on the $x$ axis and (discrete) frequency $n$ on the y axis.

From Lab Sheet 3 we already know how to breack a long signal into block, a.k.a. frames.

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 13: Manual Spectrogram Calculation (for COM4502-6502 only)**
    
<ul>
    <li>
        Implement a function <code>calc_SpectralPoint(xk,n)</code> which calculates a spectral point for one discrete frequency $n$ from a input frame $x[k]$, i.e. a function which implements the well-known DFT equation
    $$
    \mathrm{DFT}\{x[k]\}   =  X[n] = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}
    $$
    for one fixed $n$.
    </li>
    <li>
        Implement a very similar function <code>calc_SpectralPointWindowed(xk,n)</code> which calculates a spectral point for one discrete frequency $n$ from a input frame $x[k]$, but in addition applies a window function $w[k]$ to the frame $x[k]$, i.e. the function should calculate
    $$
    \mathrm{DFT}\{w[k] \cdot x[k]\}   =  X^{\mathrm{w}}[n] = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} w[k] x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}
    $$
    for one fixed $n$. The window should have the same length $L_{\mathrm{DFT}}$ as your frame and should be one of the windows we discussed during the lecture.
    </li>
    <li>
        The functions above only calculate one spectral value at a time. To obtain a full spectrum, implement a function <code>calc_Manitude_Spectrum()</code> which transforms every windowed frame to the frequency domain and calculates all positive frequencies, i.e. for $0 \leq n \leq L_{\mathrm{DFT}}/2+1$.
    </li>
    <li>
        Create a function <code>create_spectrogram()</code>, which splits the complete input sequence (e.g. a loaded WAVE file) into blocks of length $L_{\mathrm{DFT}}$. These may be overlapping. For each block the spectrum should be calculated using the previously implemented function <code>calc_Manitude_Spectrum()</code> and all spectra should be collected to form a spectrogram (e.g. as columns of a matrix).
    </li>
    <li>
        Concatenate the resulting spectra to a spectrogram and display the resulting spectrogram. You can use <code>matplotlib</code>'s <code><a href="https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html">imshow()</a></code> function for manually plotting the spectrogram image. Note that a spectrogram is usually shown in dB scaling.
    </li>
    <li>
        Visualise the input signal $x[k]$ as a spectrogram for (i) a speech signal and (ii) for a chirp/sweep signal.
    </li>
</ul>
</div>

Implement the  DFT equation
    $$\mathrm{DFT}\{x[k]\}   =  X[n] = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}$$ for one fixed $n$:

In [ ]:
def calc_SpectralPoint(xk,n):
    '''
    Implementation of the Discrete Fourier Transform (DFT).
    Calculates the Fourier coefficient X[n] for one discrete frequency n

    Input:
    xk:     time domain signal vector
    n:      discrete frequency to be calculated

    Output
    Xn : discrete frequency domain point for frequency n
    '''

    # Your code here
    # ...

Implement DFT of windowed frame
$$\mathrm{DFT}\{w[k] \cdot x[k]\}   =  X^{\mathrm{w}}[n]  = \frac{1}{L_{\mathrm{DFT}}} \sum \limits_{k=0}^{L_{\mathrm{DFT}}-1} w[k] x[k]  e^{j 2 \pi k n /L_{\mathrm{DFT}}}$$ for one fixed $n$.

In [ ]:
def calc_SpectralPointWindowed(xk,n,window=False):
    '''
    Implementation of the Discrete Fourier Transform (DFT).
    Calculates the Fourier coefficient X[n] for one discrete frequency n

    Input:
    xk:     time domain signal vector
    n:      discrete frequency to be calculated
    window: (optional): can be False (no window) or a window name from
            the list of available numpy window functions, e.g.
            np.hamming, np.bartlett, np.blackman, np.hanning, np.kaiser
            type: function
            (feel free to implement the window differently)

    Output
    Xn : dicrete frequency domain point for frequency n
    '''
    # Your code here
    # L_DFT = ???
    # ...

    if window == False:
        win = np.ones(L_DFT) # this is a window with no effect
    else:
        None # replace this by your own window

    # Your code here
    # ...

The following function <code>calc_Manitude_Spectrum()</code> is supposed to transform every (windowed or not windowed) frame to the frequency domain and to calculate all positive frequencies, i.e. $X[n]$ or $X^{\mathrm{w}}[n]$ for $0 \leq n \leq L_{\mathrm{DFT}}/2+1$.

In [ ]:
def calc_Manitude_Spectrum(xk):
    '''
    Compute Fourier coefficients up to the Nyquest Limit (fs/2), i.e. Xn for n=0,...,L_DFT/2
    using one of the two functions created before.
    and multiply the absolute value of the Fourier coefficients by 2,
    to account for the symmetry of the Fourier coefficients above the Nyquest Limit.
    '''
    # Your code here
    # ...

    # probably there should be a loop over n here

The following function <code>create_spectrogram()</code> should calculate all spectra needed for your spectrogram.

In [ ]:
def create_spectrogram(x, L_DFT=512, noverlap):
    '''
           x: original time series
       L_DFT: The number of data points used in each block for the DFT. The default value is 512.
    noverlap: The number of points of overlap between blocks. The default value is 256.
    '''
    # Your code here
    # ...



The following function can be used to actually display the created spectrogram.

In [ ]:
def plot_spectrogram( #...

The following code actually calculates and plots your spectrogram (for the two signals mentioned above). Feel free to adapt parameters `L_DFT` and `noverlap`

In [ ]:
# load or create signal

# create and plot spectrogram (generated using your functions above)
L_DFT    = 256 # DFT length
noverlap = 84  # number of overlapping samples
starts, spec = create_spectrogram( #...
plot_spectrogram(#...

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 14: Apply Short-Time Fourier Transform (STFT)**
    
<ul>
    <li>
        Explain the purpose of the Short-Time Fourier Transform (STFT) and how it differs from a regular Fourier Transform. Provide a brief(!) explanation below.
    </li>
    <li>
        Apply the STFT to the signal generated in Task T2 using a window size of $256$ samples with $50$% overlap (COM4502-6502 students should use the previously created code (if Task T13 was completed), existing functions can be used by COM3502 students (or if COM4502-6502 students did not complete Task T13).
    </li>
    <li>
        Interpret the spectrogram. What information does it provide about the signal? Briefly(!) explain what you see in the spectrogram and how it represents the frequency content over time.
    </li>
</ul>
</div>

In [ ]:
# Your code here
# ...

**Answer to question in Task T14:**

<span style="font-weight:bold;color:orange">
    ...your answer here ...
</span>

## Pitch analysis

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 15: Pitch analysis**
    
<ul>
    <li>
         Extract and plot the pitch contour of a speech signal over time. Write code to estimate the pitch for each frame using an autocorrelation-based pitch detection algorithm. Briefly discuss the variations in pitch. What can pitch tell us about the speaker’s speech patterns? Provide a brief analysis of the pitch contour.
    </li>
</ul>
</div>

In [ ]:
# Your code here
# ...

**Answer to question in Task T15:**

<span style="font-weight:bold;color:orange">
    ...your answer here ...
</span>

# Speech Synthesis

<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task 16: Synthesize a simple speech sound**
    
<ul>
    <li>
        Generate a vowel sound (e.g., “ah”) using a source-filter model, where a glottal pulse train with a frequency of 120 Hz is filtered by a vocal tract filter.
        Implement the glottal pulse train as a periodic signal and apply a simple formant-based filter. Plot the resulting waveform.
        <li>
            Hint: Use the following formant frequencies <code>formant_freqs = [730, 1090, 2440]</code> to create filters with bandwiths of <code>bandwidths = [80, 90, 120]</code>.
        </li>
    </li>
    <li>
        Plot the spectrogram of the synthesized sound. Compare it to the spectrogram of a real speech sample (which you can record yourself) containing the same vowel sound.
        Discuss any differences observed between synthetic and real speech.
    </li>
    <li>
        Briefly(!) discuss any differences observed between synthetic and real speech.
    </li>
</ul>
</div>

In [ ]:
fundamental_freq = 120             # Fundamental frequency in Hz for the glottal pulse
formant_freqs = [730, 1090, 2440]  # F1, F2, F3 for "ah" in Hz
bandwidths = [80, 90, 120]         # Bandwidths in Hz


# Your code here
# ...

**Answer to question in Task T16:**

<span style="font-weight:bold;color:orange">
    ...your answer here  (in differences observed between synthetic and real speech)...
</span>

# Equaliser (for COM4502-6502 only)

We want to design an equaliser like shown in the picture below as a hardware system.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/Yamaha_EQ-500_Graphic_Equalizer.jpg/1920px-Yamaha_EQ-500_Graphic_Equalizer.jpg" align="center" style="width: 500px;"/>
<center><span style="font-size:smaller">
    Picture taken from <a href="https://simple.wikipedia.org/wiki/Equalization_(audio)">Wikipedia</a>, license: <a href="https://creativecommons.org/licenses/by/2.0/">CC BY 2.0</a>
</span></center>


The following function realises one of the sliders in software.

In [ ]:
def peaking_filter(gain,center_freq,q,fs):
    """
    Derive coefficients for a peaking filter with a given amplitude and
     bandwidth.  All coefficients are calculated as described in Zölzer's
     DAFX book (ISBN: 0-471-49078-4, p. 50 - 55).  This algorithm assumes
     a constant q-term is used through the equation.

    Usage:     `b,a` = peaking_filter(gain,center_freq, q,fs)
                `gain` is the logarithmic gain (in dB)
                `center_freq` is the center frequency
                `q` is q-term equating to (Fb / Fc)
                `fs` is the sampling rate

    Author:    Jeff Tackett 08/22/05
    Port to Python by George Close 10/07/21
    """

    gain = np.float32(gain)
    k = np.tan((np.pi*center_freq)/fs)
    V0 = 10**((gain)/20)
    # invert gain if a cut
    if V0 < 1:
        V0 = 1/V0

    # Boost
    if gain > 0:
        b0 = (1 + ((V0/q)*k)+ k**2) / (1+((1/q)*k)+k**2)
        b1 = (2 * (k**2 - 1)) / (1 + ((1/q)*k) + k**2)
        b2 = (1 - ((V0/q)*k) + k**2) / (1 + ((1/q)*k) + k**2)
        a1 = b1
        a2 =  (1 - ((1/q)*k) + k**2) / (1 + ((1/q)*k) + k**2)
    # Cut
    elif gain <0:
        b0 = (1 + ((1/q)*k) + k**2) / (1 + ((V0/q)*k) + k**2)
        b1 =       (2 * (k**2 - 1)) / (1 + ((V0/q)*k) + k**2)
        b2 = (1 - ((1/q)*k) + k**2) / (1 + ((V0/q)*k) + k**2)
        a1 = b1
        a2 = (1 - ((V0/q)*k) + k**2) / (1 + ((V0/q)*k) + k**2)
    #gain is 0
    else:
        b0 = V0
        b1 = 0
        b2 = 0
        a1 = 0
        a2 = 0
    a = [  1, a1, a2]
    b = [ b0, b1, b2]
    return b,a

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T17: (for COM4502-6502 only)**
    
* Visualise the frequency response of one filter.
* Implement a cascade of filters to realise an equaliser.
* Visualise the frequency response of your equaliser filter and the input and (filtered) output signal.
    
</div>

In [ ]:
# Your code here
#
# ...

# Prepare for submission

<br>
<div style="border: 2px solid #999; padding: 10px; background: rgba(64, 71, 90, 1);">
    
**Task T18:**

* Clear all cell outputs to reduce the file size (in Jupyter Notebooks click on "Cell->All Output->Clear")
* Create a `.zip` file named `YourName.zip` containing this Jupyter Notebook files as well as all other files necessary to run this notebook (**if such exist**, e.g. if you created (additional) WAVE files).
* Hand in your `.zip` file via Blackboard.
    

<span style="font-weight:bold;color:red;text-align:center;">**Important: For marking, we expect your code to work ‘out of the box’.**</span> This means that no additional software should need to be installed to make the Notebook run. If you only used libraries known from the Speech Processing Lab classes, you should be safe here.
    
</div>